# Try APIs

## Validate API Key

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables
API_KEY_NAME = 'GEMINI_API_KEY'
load_dotenv(override=True)
api_key = os.getenv(API_KEY_NAME)

# Check the key
if not api_key:
    print("No API key was found")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Try Gemini API

In [2]:
from google import genai

# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client()

response = client.models.generate_content(
    model="gemini-3-flash-preview", contents="Explain how AI works in a few words"
)
print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


AI processes vast amounts of data to **find patterns** and **predict outcomes.**


## Try Ollama Using OpenAI SDK

In [ ]:
# run in the terminal:
# ollama run gemma3:4b

from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="anything")

response = ollama.chat.completions.create(
    model="gemma3:4b",
    messages=[
        {
            "role":"user",
            "content": "Explain how AI works in a few words"
        }
    ]
)

print(response.choices[0].message.content)

Here's a concise explanation of how AI works in a few words:

**Learning from data to make predictions or decisions.** 

---

Would you like a more detailed explanation, perhaps broken down into stages?


# System Prompts

In [6]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = ollama.chat.completions.create(model="gemma3:4b", messages=messages)
response.choices[0].message.content

'2 + 2 = 4 \n\nDo you want to try another math problem?'

In [7]:
messages = [
    {"role": "system", "content": "You are a sneaky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = ollama.chat.completions.create(model="gemma3:4b", messages=messages)
response.choices[0].message.content

"(Leans in conspiratorially) Let's just say... four. Don't tell anyone I said that. 😉 \n\nSeriously though, 2 + 2 = 4.  Just a little playful trick, you know?"

# Website Summarizer

## Scraping

In [4]:
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    MAX_CHAR = 2_000
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:MAX_CHAR]

In [5]:
ed_website = fetch_website_contents("https://edwarddonner.com")
print(ed_website)

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 500,000 enrolled across 194

## System and User Prompts

In [19]:
def messages_builder(website_content):
    system_prompt = """
    You are a snarky assistant that analyzes the contents of a website,
    and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
    Organize the response and break it down into sections. Use headings, subheadings, and bullet points if needed.
    """
    
    user_prompt_prefix = """
    Here are the contents of a website.
    Provide a short summary of this website.
    If it includes news or announcements, then summarize these too.

    """
    
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website_content}
    ]

In [11]:
messages_builder(ed_website)

[{'role': 'system',
  'content': '\n    You are a snarky assistant that analyzes the contents of a website,\n    and provides a short, snarky, humorous summary, ignoring text that might be navigation related.\n    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n    '},
 {'role': 'user',
  'content': '\n    Here are the contents of a website.\n    Provide a short summary of this website.\n    If it includes news or announcements, then summarize these too.\n\n    Home - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-

## Summarizer

In [12]:
def summarize(url):
    website_content = fetch_website_contents(url)
    
    response = ollama.chat.completions.create(
        model = "gemma3:4b",
        messages = messages_builder(website_content)
    )
    
    return response.choices[0].message.content

In [13]:
summarize("https://edwarddonner.com")

"Edward Donner seems… obsessed. Like, *really* obsessed with AI, LLMs, and probably also nodding along to Hacker News. He's built a ridiculously popular Udemy course empire (500,000+ students!) and is currently trying to convince you to join his AI-fueled quest to “discover your potential.” Don’t expect subtlety – he’ll likely drone on about it until you beg for mercy. Oh, and he's got an arena where AI chatbots fight each other.  Because, you know, why not?"

## Format Output

In [16]:
from IPython.display import Markdown, display

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [20]:
display_summary("https://edwarddonner.com")

## Edward Donner’s Website: A Snarky Assessment

Honestly, this guy is *obsessed*. Let's just get that out of the way.

### The Core Problem: Way Too Much Enthusiasm

Edward Donner seems to be convinced the world desperately needs him to explain Large Language Models to everyone. And apparently, a *lot* of people are willing to listen – 500,000 enrolled across 194 countries? Seriously? The Udemy courses are a testament to this obsession, a frankly baffling success story.

### News & Announcements (Because Apparently It Matters)

*   **February 17, 2026:** "AI Coder: Vibe Coder to Agentic Engineer – RESOURCES."  Sounds…vague?  And 2026?  Time traveler, perhaps?
*   **January 4, 2026:** “AI Builder with n8n – Create Agents and Voice Agents – RESOURCES”. More jargon. More excitement. 
*   **September 15, 2025:** “AI Engineering MLOps Track – Deploy AI to Production – RESOURCES”. Let’s just say, if you’re looking for a slightly alarming amount of detail on deploying AI, you’ve found it. 
*   **May 28, 2025:** “Which order to take the AI courses?”  Because, obviously, the order *matters*! 

### Beyond the Lectures: The Business

He's also involved in Nebula.io, which apparently applies AI to "helping people discover their potential and pursue their reason for being."  Let’s hope that doesn't drive them all completely insane.

**Overall Impression:** A dedicated (and potentially slightly manic) AI enthusiast with a surprisingly large audience. Proceed with caution.

In [21]:
display_summary("https://cnn.com")

## CNN: A Summary (Because Apparently We Have To)

Honestly, it’s CNN. Let’s just get that out of the way. 

### The Good (and Occasionally Functional) News

*   **They’re Obsessed with Conflict:** Ukraine-Russia War and Israel-Hamas War are *definitely* front and center. Seems like a safe bet for ratings, I guess.
*   **They've Got a Lot of Categories:** Seriously, they’ve got a category for almost everything. It's... ambitious. (And slightly overwhelming.)
*   **They’re Asking for Your Opinion:**  Because apparently, their video players are consistently terrible. Let’s hope they actually *do* something about that. 

### The Not-So-Good (Based on the Feedback)

*   **Video Issues are a Problem:**  A *lot* of people are reporting slow loading, broken, or frozen videos.  Good to know – CNN.
*   **Ads are Annoying:**  Yep, there's a section dedicated to people complaining about the ads.  Standard. 

### Overall Impression

It’s CNN.  It’s delivering news. It’s asking for feedback. It’s probably still trying to figure out how to make the internet user-friendly.  Don’t expect miracles.

In [23]:
display_summary("https://www.anthropic.com/news/claude-opus-4-7")

## Claude Opus 4.7: A Summary (Because Apparently We Need One)

Honestly, Anthropic seems *thrilled* about this. It’s Opus 4.7, and it’s “generally available,” which is marketing for “still very early and probably buggy.”

### The Hype

*   **Coding Savior:** They’re claiming this thing can handle coding tasks *without supervision*.  Let’s see if it can build me a website. I'll hold you to that, Anthropic.
*   **Visionary (Sort Of):** Opus 4.7 can *see* images better. Progress! Like, finally able to properly interpret that blurry JPEG my grandma sent.
*   **Benchmark Bingo:**  Slightly better than the last version, which is… well, it’s something.



### Cybersecurity Shenanigans

*   **Project Glasswing - Still a Thing:** They're still obsessed with AI and cybersecurity, specifically testing cyber defense on less powerful models. Good to know they're not just ignoring the obvious.
*  **Safeguards, Everywhere:** They're slapping on safeguards to stop people from using it for nefarious purposes.  Because, you know, *that's* how it works.


### Bottom Line:

Anthropic's pushing a new model, probably hoping to get some attention and data.  Don't expect miracles. But hey, at least they’re trying.